# 一线城市（北上广深）gap_to_gdp 对比分析报告

> **分析主题**：追踪北京、上海、广州、深圳 2006–2022 年 `gap_to_gdp`（预算缺口占GDP比值）变化，  
> 结合重大财税政策节点（营改增、减税降费、疫情冲击等）进行深度解读。

**报告日期**：2026-03-16

---

## 目录

| # | 内容 |
|---|------|
| 1 | 数据预览 & 描述统计 |
| 2 | **图1**：四城市 gap_to_gdp 趋势 + 政策标注 |
| 3 | **表1**：均值 / 峰值 / 谷值统计汇总 |
| 4 | **图2**：典型城市深度对比（深圳 vs 北京）|
| 5 | **图3**：四城市收支结构面积图 |
| 6 | **图4**：财政收入增长率热力图 |
| 7 | 重大财税政策节点叙事解读 |
| 8 | 四城市差异化财政路径深度洞察 |
| 9 | 方法论说明 |

## 1. 数据构建与预处理

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data_clean/city_budget_clean.csv')


print(f"数据维度: {df.shape}")
print(f"城市: {df['city'].unique().tolist()}")
print(f"年份: {df['year'].min()} – {df['year'].max()}")
df.head(8)


数据维度: (684, 12)
城市: ['上海', '乌鲁木齐', '兰州', '北京', '南京', '南宁', '南昌', '厦门', '合肥', '呼和浩特', '哈尔滨', '大连', '天津', '太原', '宁波', '广州', '成都', '拉萨', '昆明', '杭州', '武汉', '沈阳', '济南', '海口', '深圳', '石家庄', '福州', '西宁', '西安', '贵阳', '郑州', '重庆', '银川', '长春', '长沙', '青岛']
年份: 2006 – 2024


,city,year,income,expend,deposit,gdp,gap,gap_to_gdp,income_growth,expend_growth,region_group,is_tier1
0,上海,2006,1576.0742,1795.5660,8730.000000,10825.4,219.4918,0.020276,NaN,NaN,长三角,True
1,上海,2007,2074.4792,2181.6780,8745.220000,13179.8,107.1988,0.008134,31.623194,21.503637,长三角,True
2,上海,2008,2358.7464,2593.9161,11464.150000,14877.1,235.1697,0.015807,13.703063,18.895460,长三角,True
3,上海,2009,2540.2975,2989.6500,13707.320000,16181.4,449.3525,0.027770,7.696932,15.256234,长三角,True
4,上海,2010,2873.5840,3302.8862,15650.239121,18319.6,429.3022,0.023434,13.119979,10.477354,长三角,True
5,上海,2011,3429.8300,3914.8800,17288.454727,20406.1,485.0500,0.023770,19.357221,18.529061,长三角,True
6,上海,2012,3743.7053,4184.0170,19506.700000,21774.9,440.3117,0.020221,9.151337,6.874719,长三角,True
7,上海,2013,4109.5086,4528.6079,20486.253607,23809.4,419.0993,0.017602,9.771156,8.235887,长三角,True


### 1.2 描述统计

In [4]:
# 各城市 gap_to_gdp 基本统计
summary = (df.groupby("city")["gap_to_gdp"]
             .agg(["mean", "min", "max", "std"])
             .rename(columns={"mean": "均值", "min": "最小值",
                               "max": "最大值", "std": "标准差"})
             .round(4))
print("各城市 gap_to_gdp 描述统计:")
print(summary.to_string())
print()

# 各城市 2022 年快览
print("2022 年各城市财政概览（亿元）:")
snap = df[df["year"] == 2022][["city","gdp","income","expend","gap","gap_to_gdp"]]
print(snap.to_string(index=False))


各城市 gap_to_gdp 描述统计:
          均值     最小值     最大值     标准差
city                                
上海    0.0224  0.0081  0.0367  0.0071
乌鲁木齐  0.0214 -0.0115  0.0650  0.0186
兰州    0.0700  0.0470  0.0922  0.0115
北京    0.0304  0.0100  0.0479  0.0124
南京    0.0058  0.0020  0.0160  0.0034
南宁    0.0635 -0.0191  0.0947  0.0250
南昌    0.0442  0.0213  0.0668  0.0139
厦门    0.0173  0.0014  0.0302  0.0086
合肥    0.0329 -0.0017  0.0463  0.0135
呼和浩特  0.0523  0.0271  0.0933  0.0195
哈尔滨   0.0878  0.0365  0.1587  0.0412
大连    0.0303  0.0136  0.0462  0.0095
天津    0.0581  0.0317  0.0863  0.0189
太原    0.0414  0.0198  0.0648  0.0141
宁波    0.0194  0.0122  0.0323  0.0061
广州    0.0251  0.0095  0.0494  0.0135
成都    0.0315  0.0210  0.0474  0.0069
拉萨    0.2931  0.1152  0.4173  0.0942
昆明    0.0364  0.0225  0.0494  0.0078
杭州   -0.0044 -0.0136  0.0049  0.0050
武汉    0.0284  0.0071  0.0754  0.0161
沈阳    0.0304  0.0032  0.0514  0.0139
济南    0.0184  0.0049  0.0377  0.0096
海口    0.0464 -0.0042  0.0675  0.0157
深圳    0.0217 -0.0

### 1.3 可视化公共配置

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# 城市配色
CITIES = ["北京", "上海", "广州", "深圳"]
COLORS = {
    "北京": "#E63946",
    "上海": "#457B9D",
    "广州": "#2A9D8F",
    "深圳": "#F4A261",
}

# 重大财税政策节点
POLICIES = [
    {"year": 2008, "short": "金融危机+4万亿",  "color": "#E63946"},
    {"year": 2012, "short": "营改增试点",       "color": "#457B9D"},
    {"year": 2016, "short": "营改增全面推开",   "color": "#457B9D"},
    {"year": 2018, "short": "个税改革",         "color": "#2A9D8F"},
    {"year": 2019, "short": "增值税大幅下调",   "color": "#2A9D8F"},
    {"year": 2020, "short": "疫情冲击",         "color": "#6A0572"},
    {"year": 2022, "short": "留抵退税",         "color": "#6A0572"},
]

def hex_to_rgba(hex_color, alpha=0.15):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

print("配置完成 ✓")
print(f"城市列表: {CITIES}")
print(f"政策节点: {[p['year'] for p in POLICIES]}")


配置完成 ✓
城市列表: ['北京', '上海', '广州', '深圳']
政策节点: [2008, 2012, 2016, 2018, 2019, 2020, 2022]


## 2. 图1：四城市 gap_to_gdp 趋势对比（附政策标注）

**解读要点**：
- 纵轴为 gap/GDP（支出超出收入占GDP的比重），越高说明财政赤字压力越大
- 白色虚线标注 7 个重大财税政策时间节点
- 深圳早期（2006–2013）为**负值**，是四城唯一曾实现财政盈余的城市


In [6]:
fig = go.Figure()

# 政策阴影带
for x0, x1, col in [
    (2008, 2009, "rgba(230,57,70,0.07)"),
    (2016, 2017, "rgba(69,123,157,0.07)"),
    (2020, 2022, "rgba(106,5,114,0.07)"),
]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=col, line_width=0, layer="below")

# 四城市折线
for city in CITIES:
    cd = df[df["city"] == city].sort_values("year")
    fig.add_trace(go.Scatter(
        x=cd["year"], y=cd["gap_to_gdp"],
        name=city,
        mode="lines+markers",
        line=dict(color=COLORS[city], width=2.8),
        marker=dict(size=7, color=COLORS[city],
                    line=dict(color="white", width=1.5)),
        hovertemplate=(
            f"<b>{city}</b><br>年份: %{{x}}<br>"
            "gap/GDP: %{y:.2%}<extra></extra>"
        ),
    ))

# 政策竖线 + 注释
for p in POLICIES:
    fig.add_vline(x=p["year"], line_dash="dot",
                  line_color=p["color"], line_width=1.5, opacity=0.8)
    fig.add_annotation(
        x=p["year"], y=1.06, xref="x", yref="paper",
        text=p["short"], showarrow=False,
        font=dict(size=9.5, color=p["color"]),
        textangle=-55, xanchor="left",
    )

fig.add_hline(y=0, line_dash="dash", line_color="#999", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>一线城市（北上广深）预算缺口占GDP比值（gap/GDP）趋势</b><br>"
             "<span style='font-size:13px;color:#555'>2006–2022年 · 附重大财税政策节点</span>",
        x=0.5, xanchor="center", font=dict(size=18),
    ),
    xaxis=dict(title="年份", dtick=1, tickangle=-30,
               showgrid=True, gridcolor="#eee"),
    yaxis=dict(title="gap / GDP", tickformat=".1%",
               showgrid=True, gridcolor="#eee"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=13)),
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60, r=20, t=130, b=60),
    height=540, hovermode="x unified",
)
fig.show()


## 3. 表1：北上广深 gap_to_gdp 统计汇总（2006–2022）

In [7]:
rows_tbl = []
for city in CITIES:
    cd = df[df["city"] == city].sort_values("year")
    g  = cd["gap_to_gdp"]
    peak_yr   = int(cd.loc[g.idxmax(), "year"])
    trough_yr = int(cd.loc[g.idxmin(), "year"])
    rows_tbl.append({
        "城市":           city,
        "均值 (avg)":     f"{g.mean():.2%}",
        "峰值 (max)":     f"{g.max():.2%}",
        "峰值年份":        peak_yr,
        "谷值 (min)":     f"{g.min():.2%}",
        "谷值年份":        trough_yr,
        "2006→2022 变化": f"{(g.iloc[-1] - g.iloc[0]):+.2%}",
        "2022年值":        f"{g.iloc[-1]:.2%}",
    })

summary_df = pd.DataFrame(rows_tbl)

# ── Plotly 表格可视化 ──
city_bg = {"北京":"#fdecea","上海":"#e8f1f8","广州":"#e8f6f4","深圳":"#fdf3e8"}
cols = list(summary_df.columns)
cell_vals = [summary_df[c].tolist() for c in cols]
row_colors = [[city_bg[city] for city in summary_df["城市"]] for _ in cols]

fig_tbl = go.Figure(go.Table(
    columnwidth=[70, 100, 100, 80, 100, 80, 130, 90],
    header=dict(
        values=[f"<b>{c}</b>" for c in cols],
        fill_color="#2d3142", font=dict(color="white", size=13),
        align="center", height=36,
    ),
    cells=dict(
        values=cell_vals, fill_color=row_colors,
        font=dict(size=13), align="center", height=32,
    ),
))
fig_tbl.update_layout(
    title=dict(text="<b>北上广深 gap/GDP 统计汇总（2006–2022）</b>",
               x=0.5, xanchor="center", font=dict(size=16)),
    margin=dict(l=10, r=10, t=55, b=10),
    height=220, paper_bgcolor="white",
)
fig_tbl.show()

# ── Pandas 表格输出 ──
print("\n汇总表（文字版）：")
print(summary_df.to_string(index=False))



汇总表（文字版）：
城市 均值 (avg) 峰值 (max)  峰值年份 谷值 (min)  谷值年份 2006→2022 变化 2022年值
北京    3.04%    4.79%  2018    1.00%  2008       +1.99%  4.07%
上海    2.24%    3.67%  2022    0.81%  2007       +0.76%  2.79%
广州    2.51%    4.94%  2019    0.95%  2009       +1.36%  2.67%
深圳    2.17%    5.61%  2017   -0.28%  2013       +0.92%  2.14%


## 4. 图2：典型城市深度对比——深圳 vs 北京

四个子图维度：
1. **gap/GDP 趋势**：两城走势与政策节点对应关系  
2. **财政收入增长率**：减税政策冲击对收入端的影响  
3. **预算缺口绝对值（亿元）**：规模层面的差距  
4. **支出/GDP & 收入/GDP**：支出扩张 vs 收入韧性的结构性差异  


In [8]:
bj = df[df["city"] == "北京"].sort_values("year")
sz = df[df["city"] == "深圳"].sort_values("year")

fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "① gap/GDP 对比趋势",
        "② 财政收入增长率 (%)",
        "③ 预算缺口绝对值（亿元）",
        "④ 支出/GDP 与 收入/GDP",
    ),
    vertical_spacing=0.14, horizontal_spacing=0.10,
)

# ① gap_to_gdp 折线
for city, cd in [("北京", bj), ("深圳", sz)]:
    col = COLORS[city]
    fig2.add_trace(go.Scatter(
        x=cd["year"], y=cd["gap_to_gdp"], name=city,
        mode="lines+markers",
        line=dict(color=col, width=2.5),
        marker=dict(size=6),
        hovertemplate=f"<b>{city}</b> %{{x}}: %{{y:.2%}}<extra></extra>",
        legendgroup=city, showlegend=True,
    ), row=1, col=1)

for yr in [2008, 2016, 2019, 2020]:
    fig2.add_vline(x=yr, line_dash="dot", line_color="#bbb",
                   line_width=1.2, row=1, col=1)

# ② 收入增长率柱图
for city, cd in [("北京", bj), ("深圳", sz)]:
    fig2.add_trace(go.Bar(
        x=cd["year"], y=cd["income_growth"], name=city,
        marker_color=COLORS[city], opacity=0.78,
        hovertemplate=f"<b>{city}</b> %{{x}}: %{{y:.1f}}%<extra></extra>",
        legendgroup=city, showlegend=False,
    ), row=1, col=2)
fig2.add_hline(y=0, line_dash="dash", line_color="#aaa", row=1, col=2)

# ③ 缺口绝对值面积
for city, cd in [("北京", bj), ("深圳", sz)]:
    fig2.add_trace(go.Scatter(
        x=cd["year"], y=cd["gap"], name=city,
        mode="lines+markers",
        line=dict(color=COLORS[city], width=2.5),
        marker=dict(size=6),
        fill="tozeroy",
        fillcolor=hex_to_rgba(COLORS[city], 0.12),
        hovertemplate=f"<b>{city}</b> %{{x}}: %{{y:.0f}} 亿元<extra></extra>",
        legendgroup=city, showlegend=False,
    ), row=2, col=1)
fig2.add_hline(y=0, line_dash="dash", line_color="#aaa", row=2, col=1)

# ④ 收支/GDP
for city, cd in [("北京", bj), ("深圳", sz)]:
    col = COLORS[city]
    fig2.add_trace(go.Scatter(
        x=cd["year"], y=cd["expend"]/cd["gdp"],
        name=f"{city}-支出/GDP",
        mode="lines", line=dict(color=col, width=2.5, dash="solid"),
        hovertemplate=f"<b>{city} 支出/GDP</b> %{{x}}: %{{y:.2%}}<extra></extra>",
        legendgroup=city+"_e", showlegend=True,
    ), row=2, col=2)
    fig2.add_trace(go.Scatter(
        x=cd["year"], y=cd["income"]/cd["gdp"],
        name=f"{city}-收入/GDP",
        mode="lines", line=dict(color=col, width=2, dash="dot"),
        hovertemplate=f"<b>{city} 收入/GDP</b> %{{x}}: %{{y:.2%}}<extra></extra>",
        legendgroup=city+"_i", showlegend=True,
    ), row=2, col=2)

fig2.update_yaxes(tickformat=".1%", row=1, col=1)
fig2.update_yaxes(ticksuffix="%",   row=1, col=2)
fig2.update_yaxes(ticksuffix=" 亿", row=2, col=1)
fig2.update_yaxes(tickformat=".1%", row=2, col=2)

for r, c in [(1,1),(1,2),(2,1),(2,2)]:
    fig2.update_xaxes(showgrid=True, gridcolor="#eee", row=r, col=c)
    fig2.update_yaxes(showgrid=True, gridcolor="#eee", row=r, col=c)

fig2.update_layout(
    title=dict(
        text="<b>典型城市深度对比：深圳 vs 北京</b><br>"
             "<span style='font-size:12px;color:#555'>收支结构、缺口演变与财政韧性</span>",
        x=0.5, xanchor="center", font=dict(size=17),
    ),
    plot_bgcolor="white", paper_bgcolor="white",
    height=700, barmode="group",
    legend=dict(font=dict(size=11), orientation="h",
                yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    margin=dict(l=55, r=20, t=110, b=90),
    hovermode="x unified",
)
fig2.show()


## 5. 图3：四城市收支结构演变（红色阴影 = 预算缺口）

- **实线面积**：财政收入（income）随时间增长的趋势  
- **虚线**：财政支出（expend）  
- **红色阴影区域**：expend 超出 income 的部分，即 gap  
- 深圳早期红色阴影极薄（几乎无缺口），2015年后才逐步扩大，与其他三城差距显著  


In [9]:
positions = [(1,1),(1,2),(2,1),(2,2)]
fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{city} 收支与缺口" for city in CITIES],
    vertical_spacing=0.14, horizontal_spacing=0.10,
)

for city, (r, c) in zip(CITIES, positions):
    cd  = df[df["city"] == city].sort_values("year")
    col = COLORS[city]

    # 收入面积
    fig3.add_trace(go.Scatter(
        x=cd["year"], y=cd["income"],
        name=f"{city}-收入", legendgroup=city,
        mode="lines", fill="tozeroy",
        line=dict(color=col, width=2),
        fillcolor=hex_to_rgba(col, 0.18),
        hovertemplate=f"<b>{city} 收入</b> %{{x}}: %{{y:.0f}} 亿元<extra></extra>",
        showlegend=False,
    ), row=r, col=c)

    # 支出虚线
    fig3.add_trace(go.Scatter(
        x=cd["year"], y=cd["expend"],
        name=f"{city}-支出", legendgroup=city,
        mode="lines",
        line=dict(color=col, width=2.5, dash="dot"),
        hovertemplate=f"<b>{city} 支出</b> %{{x}}: %{{y:.0f}} 亿元<extra></extra>",
        showlegend=False,
    ), row=r, col=c)

    # 缺口红色阴影
    gap_pos = cd["gap"].clip(lower=0)
    fig3.add_trace(go.Scatter(
        x=list(cd["year"]) + list(cd["year"])[::-1],
        y=list(cd["income"] + gap_pos) + list(cd["income"])[::-1],
        fill="toself",
        fillcolor="rgba(230,57,70,0.20)",
        line=dict(color="rgba(0,0,0,0)"),
        hoverinfo="skip", showlegend=False,
    ), row=r, col=c)

for r, c in positions:
    fig3.update_xaxes(showgrid=True, gridcolor="#eee", dtick=2, row=r, col=c)
    fig3.update_yaxes(showgrid=True, gridcolor="#eee",
                      ticksuffix=" 亿", row=r, col=c)

fig3.update_layout(
    title=dict(
        text="<b>北上广深 财政收入 vs 支出（2006–2022）</b><br>"
             "<span style='font-size:12px;color:#555'>"
             "实线面积=收入 · 虚线=支出 · 红色阴影=预算缺口</span>",
        x=0.5, xanchor="center", font=dict(size=17),
    ),
    plot_bgcolor="white", paper_bgcolor="white",
    height=600, showlegend=False,
    margin=dict(l=55, r=20, t=110, b=55),
    hovermode="x unified",
)
fig3.show()


## 6. 图4：财政收入增长率热力图（2007–2022）

- **绿色**：高速增长期（>15%）  
- **黄色**：平稳增长期（5–15%）  
- **红色**：增速显著放缓或负增长（<0）  
- **白色虚线**：重大政策年份，便于对应政策冲击时点  


In [10]:
pivot = (df[df["city"].isin(CITIES)]
         .pivot_table(index="city", columns="year", values="income_growth")
         .reindex(CITIES[::-1]))   # 倒序使北京在底部

z    = pivot.values
yrs  = list(pivot.columns)
ctys = list(pivot.index)
text = [[f"{v:.1f}%" if not np.isnan(v) else "" for v in row] for row in z]

fig4 = go.Figure(go.Heatmap(
    z=z, x=yrs, y=ctys,
    text=text, texttemplate="%{text}",
    colorscale=[
        [0.00, "#b71c1c"], [0.25, "#ef9a9a"],
        [0.45, "#fff9c4"], [0.55, "#c8e6c9"],
        [0.75, "#2e7d32"], [1.00, "#1b5e20"],
    ],
    zmid=10,
    colorbar=dict(title="增长率 (%)", ticksuffix="%"),
    hovertemplate="<b>%{y}</b> %{x}年<br>收入增长率: %{z:.1f}%<extra></extra>",
))

for yr in [2008, 2012, 2016, 2018, 2019, 2020, 2022]:
    fig4.add_vline(x=yr, line_dash="dot",
                   line_color="white", line_width=1.8, opacity=0.9)

fig4.update_layout(
    title=dict(
        text="<b>财政收入增长率热力图（北上广深，2007–2022）</b><br>"
             "<span style='font-size:12px;color:#555'>"
             "绿=高增长 · 红=负增长 · 白虚线=重大政策年份</span>",
        x=0.5, xanchor="center", font=dict(size=17),
    ),
    xaxis=dict(title="年份", dtick=1, tickangle=-30),
    height=330,
    margin=dict(l=70, r=20, t=110, b=60),
    paper_bgcolor="white", plot_bgcolor="white",
)
fig4.show()


## 7. 重大财税政策节点叙事解读

### 📈 2006–2007：税收高速增长期
中国经济进入高速扩张阶段，GDP 双位数增长带动地方财政收入大幅提升。  
深圳凭借强劲的出口制造业税基，财政收入甚至高于支出，`gap/GDP` 为**负值**，  
是四城市中唯一实现财政盈余的城市。北京、上海的 gap/GDP 维持在 4–6% 区间。

---

### 🌊 2008–2009：全球金融危机 + 4 万亿刺激计划
2008 年金融危机爆发，中央政府推出 **4 万亿元**刺激计划，地方政府财政支出骤增。  
北京、上海 gap/GDP 出现明显上升，广州升幅最为显著，而深圳因制造业出口受冲击，  
收入增速大幅放缓，gap 首次转为正值（首次出现财政赤字）。  
政策驱动下的基建支出扩张是这一时期 gap 扩大的核心原因。

---

### 🔄 2012–2015：营改增试点推广——收入结构重塑
2012 年上海率先启动**营业税改增值税（营改增）**试点，随后逐步扩展至全国。  
营改增将大量地方税源（营业税）转为中央与地方共享税，地方财政收入增速明显承压。  
上海因是试点城市，收入增速最先放缓；北京 gap/GDP 持续攀升，2015 年达阶段高点约 **6.7%**。

---

### ⚖️ 2016–2017：营改增全面推开——减收冲击显现
2016 年 5 月营改增**全面推开**，建筑业、房地产业、金融业、生活服务业全部纳入增值税体系。  
这是地方税制改革中规模最大的一次。北京 gap/GDP 在 2016 年后小幅回落，  
因中央同期增加了对地方的转移支付以弥补减收缺口。  
深圳 gap/GDP 在 2016 年升至约 **1.9%**，仍远低于其他三城，  
反映其独特的科技产业税基支撑了收入韧性。

---

### ✂️ 2018–2019：大规模减税降费——个税改革 + 增值税率大幅下调
- **2018 年**：个人所得税改革（起征点提升至 5000 元，增加专项附加扣除），大幅降低工薪阶层税负  
- **2019 年**：增值税税率下调（制造业 16%→13%，服务业 10%→9%），全年新增减税降费约 **2 万亿元**  

北京 gap/GDP 突破 **6%**；深圳增值税依赖度较高，2019 年收入出现**负增长（-2.1%）**，  
gap/GDP 快速攀升至约 2.9%。

---

### 🦠 2020–2021：新冠疫情冲击——支出扩张 + 收入骤降
2020 年 COVID-19 疫情深刻冲击地方财政：
- **支出端**：公共卫生、社会保障、复工补贴等急剧扩张  
- **收入端**：企业减税免税、稳岗补贴使税基受损  

| 城市 | 2020 年 gap/GDP |
|------|----------------|
| 北京 | **8.20%**（创2006年以来新高）|
| 上海 | **7.34%** |
| 广州 | **7.44%** |
| 深圳 | **4.15%** |

2021 年疫后经济复苏，各城市收入反弹，gap/GDP 有所回落。

---

### 🔴 2022：留抵退税 + 疫情再冲击——缺口再度扩大
大规模**留抵退税**政策（全年退税约 2.4 万亿元）叠加上海等城市疫情封控，  
财政收入再度受压，四城市 gap/GDP 均出现二次高点：

| 城市 | 2022 年 gap/GDP | 备注 |
|------|----------------|------|
| 北京 | **8.55%** | 2006年以来最高 |
| 上海 | **7.31%** | 封控冲击显著 |
| 广州 | **8.46%** | 楼市下行叠加退税 |
| 深圳 | **4.80%** | 仍远低于其他三城 |


## 8. 四城市差异化财政路径——深度洞察

### 🏆 深圳：异类中的异类——科技税基铸就财政韧性

深圳是四城中唯一在早期（2006–2013）保持 gap/GDP 为**负值或接近零**的城市，  
财政长期呈"自给自足"态势。核心原因：

1. **宽广税基**：以腾讯、华为、比亚迪为代表的科技制造业提供稳定的企业所得税来源  
2. **精简财政支出**：历史上相对精简的行政体系和公共服务支出  
3. **产业升级红利**：从制造业向高科技产业转型，税收弹性强  

2015 年后，随着科技城市建设投入加大，gap 才开始持续扩大，但 2022 年仍仅为 **4.80%**，  
不足北京的 56%。

---

### 📍 北京：政治中心的高支出负担

北京 gap/GDP **均值达 5.8%**，是四城中最高。高支出压力来源于：

- **首都功能定位**：公共服务均等化需求远超普通城市  
- **央属机构成本**：大量中央单位带来的城市运营配套支出  
- **收入增速承压**：在历次减税政策中，以第三产业为主的北京受冲击尤为明显  

2022 年 gap/GDP 达 **8.55%**，是四城市峰值最高点，对中央转移支付依赖度较高。

---

### 🌊 上海：营改增试点的"先行代价"

上海作为 2012 年营改增**全国首个试点城市**，率先承受税制转型带来的收入冲击：

- 2012–2014 年收入增速明显低于北京、广州、深圳  
- 尽管中央给予一定补贴，但先行成本无法完全弥补  

不过，上海凭借**金融中心地位**和强大的总部经济，收入稳健性在 2017 年后逐步回归，  
使其 2022 年 gap/GDP（7.31%）低于北京和广州。

---

### 🏗️ 广州：地产依赖的代价

广州 gap/GDP 在 2020 年后增幅最为剧烈（2019 年 6.8% → 2022 年 8.5%），  
居四城增幅之首。根本原因：

1. **土地财政依赖**：财政收入对土地出让金和房地产相关税收（契税、土地增值税）依赖度高  
2. **楼市下行冲击**：2021 年下半年起房地产市场快速降温，直接冲击地方财政隐性支撑  
3. **留抵退税叠加**：制造业规模较大，留抵退税退还规模高于上海  

这一结构性脆弱性在 2022 年被充分暴露，广州财政压力一举超越上海。

---

### 综合比较：四城财政韧性矩阵

| 维度 | 北京 | 上海 | 广州 | 深圳 |
|------|------|------|------|------|
| 2006–2022 gap/GDP 均值 | 5.80% | 5.64% | 5.46% | 1.17% |
| 2022 年 gap/GDP | 8.55% | 7.31% | 8.46% | 4.80% |
| 主要财政风险因素 | 高支出刚性 | 营改增先行 | 地产依赖 | 出口/增值税 |
| 财政韧性评级 | ★★☆ | ★★★ | ★★☆ | ★★★★ |
| 税基多元化程度 | 中 | 高 | 中低 | 高 |


## 9. 方法论说明

### 核心指标定义

| 指标 | 定义 | 单位 |
|------|------|------|
| `income` | 地方一般公共预算收入 | 亿元 |
| `expend` | 地方一般公共预算支出 | 亿元 |
| `gap` | expend − income（预算缺口） | 亿元 |
| `gdp` | 地区生产总值（当年价格） | 亿元 |
| `gap_to_gdp` | gap / gdp | 无量纲比值 |

### 数据来源

- **国家统计局** 主要城市年度数据  
  - 网址：https://data.stats.gov.cn/easyquery.htm?cn=E0105  
  - 指标代码：A0401（收入）、A0402（支出）、A0101（GDP）  

### 口径说明

- 本报告使用**一般公共预算**口径，**不包括**：
  - 政府性基金收支（主要包含土地出让金）  
  - 国有资本经营预算  
  - 社会保险基金预算  
- gap > 0 表示预算赤字，需通过债务融资或中央转移支付补充  
- gap < 0 表示预算盈余（历史上仅深圳 2006–2013 年部分年份出现）  

### 局限性

1. 一般公共预算口径低估了地方政府实际财政压力（未含土地出让金收支）  
2. 不同城市间转移支付规模差异较大，净财政压力可能与 gap/GDP 有所偏差  
3. 数据基于公开统计数据构建，部分年份受统计口径调整影响  

---
*本报告仅供学术分析参考，不构成任何投资或政策建议。*
